In [1]:
import io
import math
from pathlib import Path

import pandas as pd
import librosa
import soundfile as sf
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import jiwer


In [2]:
DATA_ROOT = Path("/workspace/coral_30gb")
METADATA_PATH = DATA_ROOT / "metadata.csv"
MODEL_NAME = "openai/whisper-small"
LANGUAGE = "Danish"
TASK = "transcribe"
TARGET_SR = 16000

NUM_EPOCHS = 3
LEARNING_RATE = 1e-5
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
VAL_FRACTION = 0.02
SEED = 42
MAX_EVAL_SAMPLES = 512


In [3]:
df = pd.read_csv(METADATA_PATH)
print(df["split"].value_counts())
print(len(df))
df

split
train    59711
Name: count, dtype: int64
59711


,split,id,text,audio_path,bytes
0,train,rec_30141c4db5e1ab5e60edbd662f226755,han drog til paris for yderligere at dygtiggør...,/workspace/coral_30gb/audio/train/rec_30141c4d...,437838
1,train,rec_6677f5f759f1e09a12e43bda530f77d9,senere arbejdede han i dresden og hannover,/workspace/coral_30gb/audio/train/rec_6677f5f7...,270798
2,train,rec_7686fe5ffecf9247301c3ab7f6df8730,filmen er baseret på en roman af fletcher kneb...,/workspace/coral_30gb/audio/train/rec_7686fe5f...,529998
3,train,rec_daf33e4bd6b8a7f753f831775fc2c2a5,menneskesønnen repræsenterer altså også deres ...,/workspace/coral_30gb/audio/train/rec_daf33e4b...,616398
4,train,rec_327824a75063cef4f63ac77071618b76,hvis en evakuering af stationen havde været nø...,/workspace/coral_30gb/audio/train/rec_327824a7...,483918
...,...,...,...,...,...
59706,train,rec_48e48a5076633d3c2de9b4831fd7fb4e,senere det år rejste han tilbage til tjekkoslo...,/workspace/coral_30gb/audio/train/rec_48e48a50...,553038
59707,train,rec_ff16682ef34b20777f641f02303359a4,lovecraft fanzine redigeret af thomas winther,/workspace/coral_30gb/audio/train/rec_ff16682e...,437838
59708,train,rec_cfd0614c3d229decf67f55d668eebd6b,byen ligger på jernbanelinien fra verona via b...,/workspace/coral_30gb/audio/train/rec_cfd0614c...,633678
59709,train,rec_ff3d7063dac646c1271d7c96c40119e4,med bayern nåede han at vinde fire tyske meste...,/workspace/coral_30gb/audio/train/rec_ff3d7063...,581838


In [4]:
train_df = df[df["split"] == "train"].copy()

val_df = train_df.sample(frac=VAL_FRACTION, random_state=SEED)
train_df = train_df.drop(val_df.index).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

if len(val_df) > MAX_EVAL_SAMPLES:
    val_df = val_df.sample(n=MAX_EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)

print("train rows:", len(train_df))
print("val rows:", len(val_df))


train rows: 58517
val rows: 512


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    low_cpu_mem_usage=True,
).to(device)

forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
model.generation_config.forced_decoder_ids = forced_decoder_ids
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.config.use_cache = False
model.gradient_checkpointing_enable()

decoder_start_token_id = processor.tokenizer.bos_token_id

print(device, dtype, MODEL_NAME)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

cuda torch.bfloat16 openai/whisper-small


In [6]:
def load_audio_array(audio_path):
    audio_array, sampling_rate = sf.read(audio_path)

    if getattr(audio_array, "ndim", 1) > 1:
        audio_array = audio_array.mean(axis=1)

    if sampling_rate != TARGET_SR:
        audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=TARGET_SR)
        sampling_rate = TARGET_SR

    return audio_array, sampling_rate


In [7]:
class LocalWhisperDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "id": row["id"],
            "audio": row["audio_path"],
            "sentence": row["text"],
        }

train_dataset = LocalWhisperDataset(train_df)
val_dataset = LocalWhisperDataset(val_df)

print(len(train_dataset), len(val_dataset))


58517 512


In [8]:
def collate_batch(features):
    audio_arrays = []
    label_features = []

    for feature in features:
        text = str(feature["sentence"]).strip()
        if not text:
            continue

        audio_array, _ = load_audio_array(feature["audio"])
        audio_arrays.append(audio_array)
        label_features.append({"input_ids": processor.tokenizer(text).input_ids})

    batch = processor.feature_extractor(
        audio_arrays,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
    )
    batch["input_features"] = batch["input_features"].to(dtype=dtype)

    labels_batch = processor.tokenizer.pad(label_features, return_tensors="pt")
    labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

    if decoder_start_token_id is not None and (labels[:, 0] == decoder_start_token_id).all().cpu().item():
        labels = labels[:, 1:]

    batch["labels"] = labels
    return batch


In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=2,
    pin_memory=True,
)

print("steps per epoch:", math.ceil(len(train_loader) / GRAD_ACCUM_STEPS))


steps per epoch: 1829


In [10]:
@torch.no_grad()
def evaluate_model():
    model.eval()

    predictions = []
    references = []

    for batch in val_loader:
        input_features = batch["input_features"].to(device)
        label_ids = batch["labels"].clone()

        generated_ids = model.generate(input_features=input_features, max_length=225)

        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

        pred_text = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        ref_text = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        predictions.extend(pred_text)
        references.extend(ref_text)

    wer = jiwer.wer(references, predictions)
    return {
        "wer": wer,
        "predictions": predictions[:5],
        "references": references[:5],
    }


In [11]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

use_fp16_scaler = (dtype == torch.float16 and device.type == "cuda")
scaler = torch.cuda.amp.GradScaler() if use_fp16_scaler else None


In [12]:
autocast_dtype = torch.bfloat16 if dtype == torch.bfloat16 else torch.float16

for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad()
    running_loss = 0.0
    optimizer_steps = 0

    for step, batch in enumerate(train_loader, start=1):
        input_features = batch["input_features"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(device_type="cuda", dtype=autocast_dtype, enabled=(device.type == "cuda")):
            outputs = model(input_features=input_features, labels=labels)
            loss = outputs.loss / GRAD_ACCUM_STEPS

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        running_loss += loss.item()

        if step % GRAD_ACCUM_STEPS == 0:
            if scaler is not None:
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad()
            optimizer_steps += 1

            if optimizer_steps % 20 == 0:
                print({
                    "epoch": epoch + 1,
                    "step": optimizer_steps,
                    "loss": running_loss / 20,
                })
                running_loss = 0.0

    metrics = evaluate_model()
    print({
        "epoch": epoch + 1,
        "val_wer": metrics["wer"],
    })


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


{'epoch': 1, 'step': 20, 'loss': 2.389375573396683}
{'epoch': 1, 'step': 40, 'loss': 1.577328473329544}
{'epoch': 1, 'step': 60, 'loss': 1.2433191329240798}
{'epoch': 1, 'step': 80, 'loss': 1.1440533325076103}
{'epoch': 1, 'step': 100, 'loss': 1.0480201572179795}
{'epoch': 1, 'step': 120, 'loss': 1.0047689899802208}
{'epoch': 1, 'step': 140, 'loss': 1.0011854127049447}
{'epoch': 1, 'step': 160, 'loss': 0.9862035498023033}
{'epoch': 1, 'step': 180, 'loss': 0.9159195810556412}
{'epoch': 1, 'step': 200, 'loss': 0.8524365842342376}
{'epoch': 1, 'step': 220, 'loss': 0.847070449590683}
{'epoch': 1, 'step': 240, 'loss': 0.8380159825086594}
{'epoch': 1, 'step': 260, 'loss': 0.8325177818536759}
{'epoch': 1, 'step': 280, 'loss': 0.8092324957251549}
{'epoch': 1, 'step': 300, 'loss': 0.7428988352417946}
{'epoch': 1, 'step': 320, 'loss': 0.7322253331542015}
{'epoch': 1, 'step': 340, 'loss': 0.7243525564670563}
{'epoch': 1, 'step': 360, 'loss': 0.6774992063641548}
{'epoch': 1, 'step': 380, 'loss': 0

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

{'epoch': 1, 'val_wer': 0.2713958002470443}
{'epoch': 2, 'step': 20, 'loss': 0.4487687170505524}
{'epoch': 2, 'step': 40, 'loss': 0.45790529772639277}
{'epoch': 2, 'step': 60, 'loss': 0.49806898459792137}
{'epoch': 2, 'step': 80, 'loss': 0.49062447175383567}
{'epoch': 2, 'step': 100, 'loss': 0.4721032314002514}
{'epoch': 2, 'step': 120, 'loss': 0.4675634130835533}
{'epoch': 2, 'step': 140, 'loss': 0.46025056913495066}
{'epoch': 2, 'step': 160, 'loss': 0.49724818468093873}
{'epoch': 2, 'step': 180, 'loss': 0.48047718331217765}
{'epoch': 2, 'step': 200, 'loss': 0.5049330294132233}
{'epoch': 2, 'step': 220, 'loss': 0.45731955394148827}
{'epoch': 2, 'step': 240, 'loss': 0.48040038496255877}
{'epoch': 2, 'step': 260, 'loss': 0.47393063083291054}
{'epoch': 2, 'step': 280, 'loss': 0.46127164363861084}
{'epoch': 2, 'step': 300, 'loss': 0.46605021730065344}
{'epoch': 2, 'step': 320, 'loss': 0.48162889406085013}
{'epoch': 2, 'step': 340, 'loss': 0.46838048845529556}
{'epoch': 2, 'step': 360, 'lo

In [13]:
metrics = evaluate_model()
print({"final_val_wer": metrics["wer"]})
print("Reference:", metrics["references"][0])
print("Prediction:", metrics["predictions"][0])


{'final_val_wer': 0.25304393859184754}
Reference: diabolus in musica er det syvende studiealbum af det amerikanske thrash metalband slayer
Prediction: de aplussen musiker er det syvende studiealbum af det amerikanske træs metalband slayer


In [14]:
output_dir = Path("/workspace/outputs/whisper-small-coral-30gb")
output_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

print(f"Saved to {output_dir}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /workspace/outputs/whisper-small-coral-30gb


In [27]:
import pandas as pd
from pathlib import Path

TEST_METADATA_PATH = Path("/workspace/coral_30gb/metadata_test.csv")
test_df = pd.read_csv(TEST_METADATA_PATH)

print(len(test_df))
print(test_df["split"].value_counts())
test_df.head()



1997
split
test    1997
Name: count, dtype: int64


,split,id,text,audio_path,bytes
0,test,rec_45064562a057476e51179297fb172057,normannerne er en dansk film fra 1976,/workspace/coral_30gb/audio/test/rec_45064562a...,483918
1,test,rec_becc6eb53b441b31116584cd5c4e1c51,begge klynger støttes af klyngeorganisationer ...,/workspace/coral_30gb/audio/test/rec_becc6eb53...,702798
2,test,rec_e59f62c2675fea5f9e7e726b51283250,i mildere tilfælde kan smerterne klares med pa...,/workspace/coral_30gb/audio/test/rec_e59f62c26...,777678
3,test,rec_a47501841f128c1e6eb7fdd21647aa14,grant at ødelægge lees hær gennem udmattelse v...,/workspace/coral_30gb/audio/test/rec_a47501841...,956238
4,test,rec_164d762f9438572ebe7b3bb73b2583b6,i 1999 blev herning rejser fusioneret med falk...,/workspace/coral_30gb/audio/test/rec_164d762f9...,645198


In [28]:
from torch.utils.data import Dataset, DataLoader

class LocalWhisperDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "id": row["id"],
            "audio": row["audio_path"],
            "sentence": row["text"],
        }

test_dataset = LocalWhisperDataset(test_df)
print(len(test_dataset))


1997


In [29]:
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=2,
    pin_memory=True,
)

print("test batches:", len(test_loader))


test batches: 500


In [30]:
base_model, base_processor = load_whisper_model_and_processor(BASE_MODEL_NAME)

base_test_results = evaluate_model_pair(
    model=base_model,
    processor=base_processor,
    val_loader=test_loader,
    device=device,
)

print({"base_test_wer": base_test_results["wer"]})
print("Reference:", base_test_results["references"][0])
print("Base prediction:", base_test_results["predictions"][0])


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

{'base_test_wer': 0.6313470901924415}
Reference: normannerne er en dansk film fra 1976
Base prediction:  og nu mener jeg, at denne film for 1906 er fjerns.


In [31]:
finetuned_model, finetuned_processor = load_whisper_model_and_processor(FINETUNED_DIR)

finetuned_test_results = evaluate_model_pair(
    model=finetuned_model,
    processor=finetuned_processor,
    val_loader=test_loader,
    device=device,
)

print({"finetuned_test_wer": finetuned_test_results["wer"]})
print("Reference:", finetuned_test_results["references"][0])
print("Finetuned prediction:", finetuned_test_results["predictions"][0])


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

{'finetuned_test_wer': 0.3973104567586367}
Reference: normannerne er en dansk film fra 1976
Finetuned prediction: anomandre er en dansk film fra 1976


In [32]:
base_test_wer = base_test_results["wer"]
finetuned_test_wer = finetuned_test_results["wer"]

relative_test_improvement = (
    (base_test_wer - finetuned_test_wer) / base_test_wer
    if base_test_wer > 0 else 0.0
)

print({
    "base_test_wer": base_test_wer,
    "finetuned_test_wer": finetuned_test_wer,
    "absolute_gain": base_test_wer - finetuned_test_wer,
    "relative_improvement": relative_test_improvement,
})



{'base_test_wer': 0.6313470901924415, 'finetuned_test_wer': 0.3973104567586367, 'absolute_gain': 0.23403663343380482, 'relative_improvement': 0.370694087403599}


In [33]:
rows = []
for ref, base_pred, ft_pred in zip(
    base_test_results["references"],
    base_test_results["predictions"],
    finetuned_test_results["predictions"],
):
    rows.append({
        "reference": ref,
        "base_prediction": base_pred,
        "finetuned_prediction": ft_pred,
    })

test_comparison_df = pd.DataFrame(rows)
test_comparison_path = Path("/workspace/outputs/whisper-small-coral-30gb/test_comparison.csv")
test_comparison_df.to_csv(test_comparison_path, index=False)

print(test_comparison_path)
test_comparison_df.head(10)



/workspace/outputs/whisper-small-coral-30gb/test_comparison.csv


,reference,base_prediction,finetuned_prediction
0,normannerne er en dansk film fra 1976,"og nu mener jeg, at denne film for 1906 er fj...",anomandre er en dansk film fra 1976
1,begge klynger støttes af klyngeorganisationer ...,Bav og klunger støtter sig klungorganisatione...,bevr klunger støttes af klungorganisationer ti...
2,i mildere tilfælde kan smerterne klares med pa...,I midlere tilfælde kan jeg smadre klare med P...,i midlertifaldet kan er smatten og klars med p...
3,grant at ødelægge lees hær gennem udmattelse v...,Grønt er du der laver lige her og gennem og u...,grønse er et ødelag liges her og gennem og uma...
4,i 1999 blev herning rejser fusioneret med falk...,I 1999 blev Herno Reise funktionert med Folk ...,i 1999 blev herning rejse fusioneret med folk ...
5,det blev til yderligere 10 mål i den første sæ...,"Det blev til ørelætik mål i den første sæson,...",det blev til øre letikmål i den første sæson o...
6,fyrtårnet har været byggnadsminne siden 1933,Fyre tagerne har været bygnesmænden sin nænde...,førtårn har været bygnersmænd siden 1933
7,i danmark har industrialiseringen været afgøre...,I Danmark har indåsdragisering været afgærend...,i danmark har industrialiseringen været afgøre...
8,horakova blev født milada kralova i prag,"Og hvorfor blev det for, at der blev en lille...",horåbov er blevet frejt med lærder kalovar i p...
9,på alle danske sirenestationer anvendtes samme...,og alle den syreenerstation anvendes sammen p...,og alle danske sirene stationer anvandles samm...
